Mount Google Drive dulu

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Path

In [6]:
BASE_DIR = "/content/drive/MyDrive/Stock_Analysis_AI/Data/raw_data"

LOAD SEMUA DATA

In [7]:
import os
from glob import glob

BASE_DIR = "/content/drive/MyDrive/Stock_Analysis_AI/Data/raw"  # sesuaikan

files = glob(BASE_DIR + "/**/*.xlsx", recursive=True)

print("TOTAL FILE FOUND:", len(files))
print("SAMPLE FILES:", files[:5])

TOTAL FILE FOUND: 87
SAMPLE FILES: ['/content/drive/MyDrive/Stock_Analysis_AI/Data/raw/2020/Financial Data and Ratio - Jan 2020.xlsx', '/content/drive/MyDrive/Stock_Analysis_AI/Data/raw/2020/Financial Data and Ratio - Feb 2020.xlsx', '/content/drive/MyDrive/Stock_Analysis_AI/Data/raw/2020/Financial Data and Ratio - Mar 2020.xlsx', '/content/drive/MyDrive/Stock_Analysis_AI/Data/raw/2020/Financial Data and Ratio - Apr 2020.xlsx', '/content/drive/MyDrive/Stock_Analysis_AI/Data/raw/2020/Financial Data and Ratio - May 2020.xlsx']


In [9]:
import os
import pandas as pd
from glob import glob

In [10]:
files = glob("/content/drive/MyDrive/Stock_Analysis_AI/Data/raw/**/*.xlsx", recursive=True)

print(len(files))

87


In [11]:
df_list = []

for f in files:
    df = pd.read_excel(f, skiprows=3)

    df["source_file"] = f
    df["year"] = f.split("/")[-2]

    df_list.append(df)

df = pd.concat(df_list, ignore_index=True)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

CLEAN DATA

In [12]:
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [13]:
df = df.loc[:, ~df.columns.str.contains("unnamed")]

In [14]:
df = df[[
    "stock_name",
    "sector",
    "sub_sector",
    "fs_date",
    "source_file",
    "year",
    "profit_for_the_period",
    "profit_attr.to_owner's"
]]

In [15]:
df["profit_for_the_period"] = (
    df["profit_for_the_period"]
    .astype(str)
    .str.replace(",", "")
)

df["profit_for_the_period"] = pd.to_numeric(
    df["profit_for_the_period"],
    errors="coerce"
)

In [16]:
df["fs_date"] = pd.to_datetime(df["fs_date"], errors="coerce")

In [17]:
df = df.dropna(subset=["stock_name", "profit_for_the_period"])

In [18]:
df = df.drop_duplicates(subset=["stock_name", "year", "fs_date"])

MERGE HISTORICAL (SORTING TIME)

In [19]:
df = df.sort_values(["stock_name", "year", "fs_date"])

FEATURE ENGINEERING

In [20]:
df["profit_growth"] = df.groupby("stock_name")["profit_for_the_period"].pct_change()

In [21]:
df_hist = df.groupby(["stock_name", "year"]).agg({
    "profit_for_the_period": "mean",
    "profit_growth": "mean",
    "sector": "first",
    "sub_sector": "first"
}).reset_index()

SIMPAN PROCESSED DATA

In [22]:
os.makedirs("Stock_Analysis_AI/processed", exist_ok=True)
df.to_csv("Stock_Analysis_AI/processed/merged_clean.csv", index=False)

FEATURE SET

In [24]:
print(df.columns)

Index(['stock_name', 'sector', 'sub_sector', 'fs_date', 'source_file', 'year',
       'profit_for_the_period', 'profit_attr.to_owner's', 'profit_growth'],
      dtype='object')


In [25]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "_")
    .str.replace("'", "")
)

In [26]:
df.dtypes

,0
stock_name,object
sector,object
sub_sector,object
fs_date,datetime64[ns]
source_file,object
year,object
profit_for_the_period,float64
profit_attr_to_owners,object
profit_growth,float64


In [28]:
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

In [29]:
df["profit_attr_to_owners"] = (
    df["profit_attr_to_owners"]
    .astype(str)
    .str.replace(",", "")
)

df["profit_attr_to_owners"] = pd.to_numeric(
    df["profit_attr_to_owners"],
    errors="coerce"
)

In [30]:
df.dtypes

,0
stock_name,object
sector,object
sub_sector,object
fs_date,datetime64[ns]
source_file,object
year,Int64
profit_for_the_period,float64
profit_attr_to_owners,float64
profit_growth,float64


SPLIT DATA (TIME SERIES STYLE)

In [ ]:
from sklearn.model_selection import train_test_split

In [32]:
df = df.sort_values(["stock_name", "year"])

In [33]:
train = df[df["year"] <= 2023]
test  = df[df["year"] > 2023]

In [34]:
print("TRAIN YEARS:", train["year"].unique())
print("TEST YEARS:", test["year"].unique())

TRAIN YEARS: <IntegerArray>
[2021, 2022, 2023, 2019, 2020]
Length: 5, dtype: Int64
TEST YEARS: <IntegerArray>
[2024, 2025, 2026]
Length: 3, dtype: Int64


In [35]:
features = ["profit_for_the_period", "profit_growth"]

X_train = train[features]
y_train = train["profit_growth"]

X_test = test[features]
y_test = test["profit_growth"]

In [38]:
train = df[df["year"] <= df["year"].quantile(0.8)]
test  = df[df["year"] > df["year"].quantile(0.8)]

In [39]:
train = df[df["year"] <= 2023]
test = df[df["year"] > 2023]

In [40]:
print("TRAIN shape:", train.shape)
print("TEST shape:", test.shape)

TRAIN shape: (10869, 9)
TEST shape: (11863, 9)


In [42]:
def sanity(df, name):
    print(f"\n{name}")
    print(df[["year", "stock_name", "profit_for_the_period"]].head())
    print(df.dtypes)

sanity(train, "TRAIN")
sanity(test, "TEST")


TRAIN
       year         stock_name  profit_for_the_period
20457  2021  ABM Investama Tbk                -112.89
21899  2021  ABM Investama Tbk                -532.32
24851  2021  ABM Investama Tbk                 429.84
26348  2021  ABM Investama Tbk                 897.27
28632  2021  ABM Investama Tbk                1557.02
stock_name                       object
sector                           object
sub_sector                       object
fs_date                  datetime64[ns]
source_file                      object
year                              Int64
profit_for_the_period           float64
profit_attr_to_owners           float64
profit_growth                   float64
dtype: object

TEST
       year         stock_name  profit_for_the_period
29407  2024  ABM Investama Tbk                3923.96
32181  2024  ABM Investama Tbk                4872.92
33112  2024  ABM Investama Tbk                 803.19
37806  2024  ABM Investama Tbk                1694.93
40652  2025  ABM In

SCALING (NN ONLY)

In [43]:
features = [
    "profit_for_the_period",
    "profit_attr_to_owners",
    "profit_growth"
]

In [44]:
X_train = train[features]
X_test = test[features]

In [46]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [47]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [48]:
import numpy as np

print("TRAIN mean:", np.mean(X_train_scaled, axis=0))
print("TRAIN std :", np.std(X_train_scaled, axis=0))

TRAIN mean: [-1.56895995e-17             nan             nan]
TRAIN std : [ 1. nan nan]


In [51]:
df = df.dropna(subset=[
    "profit_for_the_period",
    "profit_growth"
])

In [52]:
features = [
    "profit_for_the_period",
    "profit_growth"
]

X_train = train[features]
X_test = test[features]

In [53]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

X_train = X_train.dropna()
X_test = X_test.dropna()

In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [55]:
print("X_train shape:", X_train.shape)
print("X_train_scaled shape:", X_train_scaled.shape)

print("X_test shape:", X_test.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train shape: (9949, 2)
X_train_scaled shape: (9949, 2)
X_test shape: (11794, 2)
X_test_scaled shape: (11794, 2)


In [57]:
import numpy as np

print("NaN train:", np.isnan(X_train_scaled).sum())
print("NaN test:", np.isnan(X_test_scaled).sum())

print("INF train:", np.isinf(X_train_scaled).sum())
print("INF test:", np.isinf(X_test_scaled).sum())

NaN train: 0
NaN test: 0
INF train: 0
INF test: 0


MODEL XGBOOST

In [59]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [60]:
df = df.sort_values(["stock_name", "year"])

In [61]:
df["target"] = df.groupby("stock_name")["profit_growth"].shift(-1)

In [62]:
df = df.dropna(subset=["target"])

In [63]:
features = [
    "profit_for_the_period",
    "profit_attr_to_owners",
    "profit_growth"
]

In [64]:
train = df[df["year"] <= 2023]
test  = df[df["year"] > 2023]

X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

In [65]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [66]:
y_pred = model.predict(X_test)

In [67]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 73.46996264390573
R2 Score: -0.03973261741368672


In [68]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 73.46996264390573
R2 Score: -0.03973261741368672


In [69]:
test_result = test.copy()
test_result["prediction"] = y_pred

ranking = test_result.groupby("stock_name")["prediction"].mean().sort_values(ascending=False)

print(ranking.head(10))

stock_name
Repower Asia Indonesia Tbk.           82.809235
Tira Austenite Tbk                    63.519009
Mustika Ratu Tbk                      60.412754
Wahana Inti Makmur Tbk.               57.113552
PT Logisticsplus International Tbk    34.010715
Nanotech Indonesia Global Tbk.        33.705143
Era Mandiri Cemerlang Tbk.            32.118160
Indo Straits Tbk.                     31.935453
PT Sumber Sinergi Makmur Tbk          31.446930
Diagnos Laboratorium Utama Tbk.       29.395498
Name: prediction, dtype: float32


NEURAL NETWORK

In [70]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras

In [71]:
df = df.sort_values(["stock_name", "year"])

In [72]:
df["target"] = df.groupby("stock_name")["profit_growth"].shift(-1)
df = df.dropna(subset=["target"])

In [73]:
features = [
    "profit_for_the_period",
    "profit_attr_to_owners",
    "profit_growth"
]

In [74]:
train = df[df["year"] <= 2023]
test  = df[df["year"] > 2023]

X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

In [75]:
X_train = X_train.replace([np.inf, -np.inf], np.nan).dropna()
X_test = X_test.replace([np.inf, -np.inf], np.nan).dropna()

y_train = y_train.loc[X_train.index]
y_test = y_test.loc[X_test.index]

In [76]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [77]:
model = keras.Sequential([
    keras.layers.Dense(64, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [78]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 391.6315 - val_loss: 19.4763
Epoch 2/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 391.6478 - val_loss: 19.5084
Epoch 3/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 391.6279 - val_loss: 19.4667
Epoch 4/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 391.6164 - val_loss: 19.4639
Epoch 5/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 391.6158 - val_loss: 19.5298
Epoch 6/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 391.6221 - val_loss: 19.4423
Epoch 7/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 391.6233 - val_loss: 19.4750
Epoch 8/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 391.6160 - val_loss: 19.5120
Epoch 9/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 391.6455 - val_loss: 19.4873
Epoch 10/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 391.6195 - val_loss: 19.4725
Epoch 11/50
249/249 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 391.6182 - val_loss: 19.5002
Epoch 12/50
249/249

In [79]:
y_pred = model.predict(X_test_scaled).flatten()

308/308 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [80]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 72.41650358775844
R2: -6.083291617331454e-06


In [81]:
test_result = test.loc[X_test.index].copy()
test_result["prediction"] = y_pred

ranking = test_result.groupby("stock_name")["prediction"].mean().sort_values(ascending=False)

print(ranking.head(10))

stock_name
MDS Retailing Tbk.                                0.240925
Ultrajaya Milk Industry & Trading Company Tbk.    0.234509
Media Nusantara Citra Tbk.                        0.233793
Siantar Top Tbk.                                  0.232862
Citra Marga Nusaphala Persada Tbk.                0.228926
Gajah Tunggal Tbk.                                0.227382
Industri Jamu dan Farmasi Sido Muncul Tbk.        0.224533
Bank BTPN Syariah Tbk.                            0.224071
PT Sawit Sumbermas Sarana Tbk.                    0.222407
Avia Avian Tbk.                                   0.222063
Name: prediction, dtype: float32


SAVE MODEL

In [83]:
import joblib

joblib.dump(model, "xgboost_model.pkl")
print("XGBoost model saved!")

XGBoost model saved!


In [84]:
model_xgb = joblib.load("xgboost_model.pkl")

In [85]:
model.save("nn_model.h5")
print("NN model saved!")

NN model saved!


In [88]:
model.save("nn_model.keras")

In [89]:
from tensorflow import keras

nn_model = keras.models.load_model("nn_model.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [92]:
import joblib

joblib.dump(scaler, "scaler.pkl")
print("Scaler saved!")

Scaler saved!


In [95]:
import os

print(os.listdir())

['.config', 'drive', 'Stock_Analysis_AI', 'scaler.pkl', 'xgboost_model.pkl', 'nn_model.h5', 'nn_model.keras', 'sample_data']


In [96]:
import joblib

joblib.dump(
    scaler,
    "/content/drive/MyDrive/Stock_Analysis_AI/model/scaler.pkl"
)

print("Scaler disimpan ke Drive!")

Scaler disimpan ke Drive!


In [97]:
import joblib

scaler = joblib.load(
    "/content/drive/MyDrive/Stock_Analysis_AI/model/scaler.pkl"
)

In [98]:
import pandas as pd
import numpy as np

# =========================
# 1. PILIH FEATURES MCDA
# =========================
mcda_features = [
    "profit_for_the_period",
    "profit_attr_to_owners",
    "profit_growth"
]

# =========================
# 2. CLEAN DATA
# =========================
df_mcda = df.dropna(subset=mcda_features).copy()

# buang inf kalau ada
df_mcda = df_mcda.replace([np.inf, -np.inf], np.nan)
df_mcda = df_mcda.dropna(subset=mcda_features)

# =========================
# 3. NORMALISASI MIN-MAX (0-1)
# =========================
norm_df = df_mcda.copy()

for col in mcda_features:
    norm_df[col] = (
        (df_mcda[col] - df_mcda[col].min()) /
        (df_mcda[col].max() - df_mcda[col].min())
    )

# =========================
# 4. BOBOT (BISA DIUBAH)
# =========================
weights = {
    "profit_for_the_period": 0.3,
    "profit_attr_to_owners": 0.3,
    "profit_growth": 0.4
}

# =========================
# 5. HITUNG MCDA SCORE
# =========================
norm_df["mcda_score"] = (
    norm_df["profit_for_the_period"] * weights["profit_for_the_period"] +
    norm_df["profit_attr_to_owners"] * weights["profit_attr_to_owners"] +
    norm_df["profit_growth"] * weights["profit_growth"]
)

# =========================
# 6. RATA-RATA PER SAHAM
# =========================
ranking = norm_df.groupby("stock_name")["mcda_score"].mean().reset_index()

# =========================
# 7. TAMBAH RISK (VOLATILITY SCORE)
# =========================
risk = norm_df.groupby("stock_name")["mcda_score"].std().reset_index()
risk.columns = ["stock_name", "risk"]

ranking = ranking.merge(risk, on="stock_name", how="left")

# =========================
# 8. FINAL RANKING (BEST FIRST)
# =========================
ranking = ranking.sort_values("mcda_score", ascending=False)

# =========================
# 9. OUTPUT TOP 10
# =========================
print("🔥 TOP 10 STOCK RANKING (MCDA)")
print(ranking.head(10))

# =========================
# 10. SAVE HASIL
# =========================
ranking.to_excel("mcda_ranking_result.xlsx", index=False)
print("✅ Saved: mcda_ranking_result.xlsx")

🔥 TOP 10 STOCK RANKING (MCDA)
                                 stock_name  mcda_score      risk
597  PT Bank Rakyat Indonesia (Persero) Tbk    0.763758  0.064852
97              Bank Mandiri (Persero) Tbk.    0.755261  0.060876
94                   Bank Central Asia Tbk.    0.751988  0.057175
60                  Astra International Tbk    0.727063  0.038945
901         Telkom Indonesia (Persero) Tbk.    0.708297  0.027141
21         Alamtri Resources Indonesia Tbk.    0.693986  0.043286
114                    Bayan Resources Tbk.    0.686163  0.031099
590  PT Bank Negara Indonesia (Persero) Tbk    0.684920  0.023030
946                    United Tractors Tbk.    0.684004  0.024058
5              Adaro Andalan Indonesia Tbk.    0.683990  0.021412
✅ Saved: mcda_ranking_result.xlsx
